# Visible–Infrared Person Re-identification on SYSU-MM01

**COMP9444 Group Project**

A PMT baseline with planned smooth-transition and local-supervision extensions.

This notebook currently reports the completed baseline training run. Retrieval evaluation and extension experiments are in progress.

## 1. Introduction, Motivation and Problem Statement

Visible–infrared person re-identification matches an infrared query image to visible-light images of the same person across cameras. Differences in spectrum, lighting, pose and occlusion make appearance matching difficult. The objective is to learn features that preserve identity across modalities and rank matching gallery images highly, including for identities unseen during training.

We use the Progressive Modality-shared Transformer (PMT) [1] as our baseline. We investigate whether a smoother grey-to-colour transition and auxiliary local supervision improve retrieval, individually and together. Performance will be assessed using mAP and Rank-1/5/10 on identity-disjoint evaluation data.

## 2. Data Sources

We use **SYSU-MM01**, supplied through the course-linked [Kaggle mirror](https://www.kaggle.com/datasets/coconutjean/sysumm01), with the official identity splits [2]. Cameras 1, 2, 4 and 5 capture RGB images; cameras 3 and 6 capture infrared images.

The dataset contains 45,863 images: 30,071 RGB and 15,792 IR. Development uses 296 training identities and 99 validation identities; 96 test identities are reserved for final evaluation. The downloaded directories contain 22 additional identities absent from the official split lists. Their 1,118 images are excluded.

In [1]:
# Saved measurements from the baseline run; no training or data download.
import json
from IPython.display import Markdown, display

def show_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

measurements = json.loads('{"images_per_split_modality": {"test/ir": 3803, "test/rgb": 6775, "train/ir": 9929, "train/rgb": 20284, "unassigned/ir": 80, "unassigned/rgb": 1038, "val/ir": 1980, "val/rgb": 1974}, "images_per_camera": {"cam1": 6289, "cam2": 7528, "cam3": 9812, "cam4": 8030, "cam5": 8224, "cam6": 5980}, "epochs": [{"epoch": 1, "stage": 1, "id": 9.895071681265206, "triplet": 2.4337873198974753, "msel": 0.0, "dcl": 0.0, "total": 12.328859010578897}, {"epoch": 2, "stage": 1, "id": 5.405836521543409, "triplet": 0.5574010249174319, "msel": 0.0, "dcl": 0.0, "total": 5.963237548891402}, {"epoch": 3, "stage": 1, "id": 2.313972295554705, "triplet": 0.27469258700014865, "msel": 0.0, "dcl": 0.0, "total": 2.588664880672714}, {"epoch": 4, "stage": 1, "id": 1.109292010654582, "triplet": 0.15504540034457984, "msel": 0.0, "dcl": 0.0, "total": 1.2643374105294545}, {"epoch": 5, "stage": 1, "id": 0.7351743248166255, "triplet": 0.11681329169771142, "msel": 0.0, "dcl": 0.0, "total": 0.8519876161693208}, {"epoch": 6, "stage": 1, "id": 0.5656185880138599, "triplet": 0.0967231511501051, "msel": 0.0, "dcl": 0.0, "total": 0.6623417392114139}, {"epoch": 7, "stage": 2, "id": 0.469481795202311, "triplet": 0.15250743363835328, "msel": 0.20957776596902106, "dcl": 0.6007366138046951, "total": 1.0271464172415259}, {"epoch": 8, "stage": 2, "id": 0.3523105321184248, "triplet": 0.09375102059639061, "msel": 0.11106696917335569, "dcl": 0.5779060389004989, "total": 0.7905480577294107}, {"epoch": 9, "stage": 2, "id": 0.27896022413508587, "triplet": 0.07092857236884192, "msel": 0.08879721369506059, "dcl": 0.5603820348708739, "total": 0.6744784196782828}, {"epoch": 10, "stage": 2, "id": 0.23269545188317836, "triplet": 0.057511916472010115, "msel": 0.07533332185546086, "dcl": 0.5446752944922937, "total": 0.600211675509834}, {"epoch": 11, "stage": 2, "id": 0.19635072999632378, "triplet": 0.04536981642795035, "msel": 0.06350961768643645, "dcl": 0.5258730547782181, "total": 0.536411883047595}, {"epoch": 12, "stage": 2, "id": 0.1853902283643989, "triplet": 0.04274636491144438, "msel": 0.057937336941418865, "dcl": 0.5124638109316382, "total": 0.5133371682716961}, {"epoch": 13, "stage": 2, "id": 0.15482122581643398, "triplet": 0.03413528978870961, "msel": 0.050652518109263966, "dcl": 0.4948721874099207, "total": 0.46171886679873836}, {"epoch": 14, "stage": 2, "id": 0.12480144161881075, "triplet": 0.029785835367147802, "msel": 0.047160174753546336, "dcl": 0.4771739614518333, "total": 0.4167543458995096}, {"epoch": 15, "stage": 2, "id": 0.1084839961635711, "triplet": 0.0266535037646282, "msel": 0.04264157512306131, "dcl": 0.4623540560781108, "total": 0.3876353150813312}, {"epoch": 16, "stage": 2, "id": 0.08627329345006866, "triplet": 0.022295507450115767, "msel": 0.03877250808269068, "dcl": 0.4441792790264503, "total": 0.35004469410973893}, {"epoch": 17, "stage": 2, "id": 0.087549884818306, "triplet": 0.022049351646285357, "msel": 0.036531552213345655, "dcl": 0.4318871846037077, "total": 0.34380860558429976}, {"epoch": 18, "stage": 2, "id": 0.07135801389519167, "triplet": 0.02020650813234661, "msel": 0.03365251646446297, "dcl": 0.4208294298694032, "total": 0.3188054945325776}, {"epoch": 19, "stage": 2, "id": 0.07076973426906101, "triplet": 0.01932831401815329, "msel": 0.032515983593077835, "dcl": 0.41057131720756845, "total": 0.3116416991416124}, {"epoch": 20, "stage": 2, "id": 0.05993384103942515, "triplet": 0.019468981734407833, "msel": 0.03045111172060125, "dcl": 0.40370109211212085, "total": 0.29647892459308933}, {"epoch": 21, "stage": 2, "id": 0.06196701410005832, "triplet": 0.017726471562653567, "msel": 0.02987099016500169, "dcl": 0.396795853054354, "total": 0.29302690770200457}, {"epoch": 22, "stage": 2, "id": 0.05798238050909845, "triplet": 0.01827493317367871, "msel": 0.029066515792202423, "dcl": 0.3944796729144327, "total": 0.28803040861706786}, {"epoch": 23, "stage": 2, "id": 0.05806540522233717, "triplet": 0.017107306881020606, "msel": 0.028732913109292917, "dcl": 0.39249476842217346, "total": 0.28578655275782516}, {"epoch": 24, "stage": 2, "id": 0.05570394186984224, "triplet": 0.018262740649433003, "msel": 0.02829119528233299, "dcl": 0.3902712081092605, "total": 0.28324788380384824}]}')

In [2]:
counts = measurements["images_per_split_modality"]
show_table(["Split", "Identities", "RGB images", "IR images"], [
    [split, ids, counts[f"{split}/rgb"], counts[f"{split}/ir"]]
    for split, ids in [("train",296), ("val",99), ("test",96), ("unassigned",22)]])

| Split | Identities | RGB images | IR images |
| --- | --- | --- | --- |
| train | 296 | 20284 | 9929 |
| val | 99 | 1974 | 1980 |
| test | 96 | 6775 | 3803 |
| unassigned | 22 | 1038 | 80 |

## 3. Exploratory Analysis of Data

### 3.1 Data balance and integrity

The training split has 20,284 RGB images and 9,929 IR images, approximately a 2:1 imbalance. We therefore sample equally from both modalities and balance identities within each training batch. All training identities have images in both modalities. The audit found no unreadable images, overlapping official identity splits or duplicate files shared across splits.

Camera-level image counts also vary, as shown below. These counts describe dataset coverage; they do not establish how difficult each camera is for retrieval.

In [3]:
show_table(["Camera", "Modality", "Images"], [
    [camera, "IR" if camera in ("cam3", "cam6") else "RGB", count]
    for camera, count in measurements["images_per_camera"].items()])

| Camera | Modality | Images |
| --- | --- | --- |
| cam1 | RGB | 6289 |
| cam2 | RGB | 7528 |
| cam3 | IR | 9812 |
| cam4 | RGB | 8030 |
| cam5 | RGB | 8224 |
| cam6 | IR | 5980 |

### 3.2 Further exploration

Per-identity sample distributions, image dimensions and intensity distributions, and representative RGB/IR and occlusion examples remain to be added with their implications for preprocessing and modelling.

## 4. Models and Methods

### 4.1 PMT baseline

We implement PMT following Lu et al. [1] and the pinned author PyTorch implementation. The shared ViT-B/16 has 12 blocks, 12 attention heads and 768-dimensional features. A 16×16 patch projection with stride 12 produces a 21×10 patch grid from 256×128 images. The CLS feature feeds a BN neck and an identity classifier. Metric losses use the pre-BN feature; retrieval uses the L2-normalised, post-BN global feature.

Training has two stages. Epochs 1–6 use greyscale RGB and IR inputs with identity classification and separate within-modality hard-triplet losses. Epochs 7–24 restore colour RGB and use joint hard-triplet, modality discrepancy (MSEL) and centre-based (DCL) losses:

$$L_1=L_{ID}^{R}+L_{ID}^{I}+L_{tri}^{R}+L_{tri}^{I}.$$
$$L_2=L_{ID}^{R}+L_{ID}^{I}+L_{tri}^{all}+0.5L_{MSEL}+0.5L_{DCL}.$$

MSEL aligns within- and cross-modality positive distances; DCL encourages identity-centred separation. We retain the stage-2 ID term as in the author's PyTorch code. PMT and its losses are prior work; our contribution here is the implementation and experiment. The augmentation recipe includes the author's random-erasing implementation. Source attribution and implementation differences are recorded in `ATTRIBUTION.md` and `docs/baseline.md`.

### 4.2 Preprocessing and training settings

Images are resized to 256×128 and normalised using ImageNet statistics. Training uses horizontal flipping and random erasing; IR additionally uses a random choice of colour jitter or Gaussian blur. Evaluation uses resize and normalisation only.

| Setting | Baseline value |
|---|---|
| Initialisation | Author-linked ImageNet ViT-B/16 weights |
| Batch | 8 identities × 4 images × 2 modalities = 64 |
| Training length | 24 epochs; first 6 in stage 1 |
| Optimiser | AdamW; weight decay 0.0001 |
| Learning rate | Base 0.0003; half rate for patch projection and transformer blocks |
| Schedule | 3-epoch warmup, cosine decay; minimum LR 0.000003 |
| Triplet margin | 0.1 |
| Executed training seed | 17 |
| Training device | NVIDIA L40S; mixed precision |

### 4.3 Proposed extensions

The smooth-transition extension gradually reduces the probability of greyscale RGB after stage 1. The local-supervision extension adds auxiliary identity losses on horizontal feature regions. The planned M0–M3 comparison varies these two factors while keeping the backbone and global retrieval feature fixed. These extensions have not yet been evaluated.

### 4.4 Retrieval protocol

Development evaluation uses IR→RGB all-search, single-shot retrieval. Ten fixed gallery draws will be selected from the RGB candidate pool. Cam2 gallery images are excluded for cam3 queries, following the SYSU protocol, and CMC is computed at identity level. Features are L2-normalised and ranked by negative cosine similarity. We will report mAP and Rank-1/5/10, averaging gallery draws within each trained model.

## 5. Results

### 5.1 Baseline training

The M0 baseline completed 24 epochs on the 296-identity training split using seed 17. The table shows mean training losses at the start and end of each stage; each row averages 633 batches. MSEL and DCL are shown before applying their 0.5 weights.

In [4]:
selected_epochs = {1, 6, 7, 24}
show_table(["Epoch", "Stage", "ID", "Triplet", "MSEL", "DCL", "Total"], [
    [row["epoch"], row["stage"], *[f"{row[key]:.4f}" for key in ["id","triplet","msel","dcl","total"]]]
    for row in measurements["epochs"] if row["epoch"] in selected_epochs])

| Epoch | Stage | ID | Triplet | MSEL | DCL | Total |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 1 | 9.8951 | 2.4338 | 0.0000 | 0.0000 | 12.3289 |
| 6 | 1 | 0.5656 | 0.0967 | 0.0000 | 0.0000 | 0.6623 |
| 7 | 2 | 0.4695 | 0.1525 | 0.2096 | 0.6007 | 1.0271 |
| 24 | 2 | 0.0557 | 0.0183 | 0.0283 | 0.3903 | 0.2832 |

Training loss decreases within both stages: from 12.3289 to 0.6623 in stage 1 and from 1.0271 to 0.2832 in stage 2. The objective and RGB input change at epoch 7, so the boundary values are not directly comparable. These measurements describe training fit; retrieval performance must be established separately.

### 5.2 Retrieval performance

The trained model has produced 768-dimensional features for 1,980 IR validation images and 1,974 RGB gallery candidates, covering 99 identities. Retrieval evaluation is pending.

| Method | Training seed | Split | mAP | Rank-1 | Rank-5 | Rank-10 |
|---|---|---|---|---|---|---|
| M0: PMT | 17 | Validation | Pending | Pending | Pending | Pending |

The completed retrieval results, training curves and extension comparisons will be added here.

## 6. Discussion

The baseline successfully learns its training objectives and provides a common starting point for the proposed extensions. Its two-stage design first learns from grey RGB/IR inputs and then introduces colour and cross-modality losses.

The current evidence is limited to one training seed and training losses. It does not yet show generalisation, superiority over another method or benefits from the proposed extensions. Retrieval metrics and difficult-query examples are needed to assess identity confusion, modality differences and occlusion failures. The current development split also differs from the final train+validation setting, so published benchmark scores should not be compared directly.

Next, we will evaluate the baseline with the fixed retrieval protocol, complete the controlled extension comparisons and revise the conclusions using their measured results.

### References

[1] Lu, H., Zou, X., & Zhang, P. (2023). *Learning Progressive Modality-shared Transformers for Effective Visible-Infrared Person Re-identification*. AAAI. [Paper](https://arxiv.org/abs/2212.00226); [author code, pinned version](https://github.com/hulu88/PMT/tree/0b31da97b06428473bc4a1134b322ea226340818).

[2] [SYSU-MM01 dataset and evaluation protocol](https://github.com/wuancong/SYSU-MM01); [course-linked data mirror](https://www.kaggle.com/datasets/coconutjean/sysumm01).